# Phase 3: train the guard candidates, on a free Colab T4

Trains each candidate base model with 3 seeds (`guard/train.py`: class-weighted loss, fp16, best
epoch by validation PR-AUC), then scores the **frozen v2 test set** with every run
(`scripts/guard_eval.py`, both halves; nothing is fitted here). docs/v2-plan.md, Phase 3 steps 3-4.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all.** Every finished run is saved
to Drive, so a disconnected session resumes: re-run all cells and finished runs are skipped.

1. GPU check, config, Drive, repo, install, Hugging Face token
2. the training set (the private Hugging Face dataset from Gate G2, or a `guardset_out.tgz` in Drive),
   checked against the build Claude reviewed
3. the runs: train → score the frozen set → save
4. `guard_results.tgz` (logs and scores, no weights) to download

Model weights stay in Drive (`weights/`), only for the seeds in `KEEP_WEIGHTS_SEEDS`.
The winner is picked on the laptop, on the **calibration half only**.

## 1 · GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader || echo "NO GPU: Runtime -> Change runtime type -> T4 GPU"

## 2 · Config

The only cell you should need to edit. **Stage A** (default): the four candidates x 3 seeds, no
ablation. **Stage B** (after the winner is picked on the laptop): `MODELS = ["<winner>"]` and
`ABLATIONS = ["en_only", "no_adv", "no_aug", "frac25", "frac50"]`.

In [ ]:
MODELS    = ["mdeberta", "muril", "indicbert2", "promptguard2"]   # guard/configs/<name>.yaml
SEEDS     = [1, 2, 3]
ABLATIONS = ["full"]
KEEP_WEIGHTS_SEEDS = [1]      # weights of these seeds go to Drive (about 0.5-1 GB each); others are scored, then deleted

# The training set. Claude fills these in at Gate G2, after the sample check; until then this
# notebook refuses to train.
DATASET_REPO        = ""      # the private Hugging Face dataset, e.g. "<user>/bhasha-shield-guardset"
DATASET_REVISION    = ""      # its commit
EXPECTED_REPORT_SHA = ""      # sha256 of the reviewed build's report.json

REPO_DIR  = "/content/agentfw"
GIT_URL   = "https://github.com/Sameerkc3579/bhasha-shield.git"
BRANCH    = "v2"
DRIVE_DIR = "/content/drive/MyDrive/agentfw_guard"
print(len(MODELS) * len(SEEDS) * len(ABLATIONS), "runs:", MODELS, "x seeds", SEEDS, "x", ABLATIONS)

## 3 · Drive

Finished runs, their scores and the kept weights live here between sessions.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/weights", exist_ok=True)
print("drive ready:", sorted(os.listdir(f"{DRIVE_DIR}/results"))[:20])

## 4 · Repo, install, token

In [ ]:
import os, subprocess, sys
if not os.path.exists(f"{REPO_DIR}/pyproject.toml"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, GIT_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT)

def run(cmd, env=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8", **(env or {})})
    for line in p.stdout:
        print(line, end="")
    return p.wait()

assert run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.46,<5", "accelerate>=0.26",
            "sentencepiece", "protobuf", "scikit-learn", "pydantic>=2.7", "pyyaml"]) == 0
assert run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."]) == 0
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN: found")
except Exception as e:
    print("HF_TOKEN: not found (" + type(e).__name__ + "): the private dataset and PromptGuard 2 need it")

## 5 · Training set

From the private Hugging Face dataset (Gate G2), or from `guardset_out.tgz` in the Drive folder.
Its `report.json` must be the build Claude reviewed (`EXPECTED_REPORT_SHA`), and its leakage audit
must be clean (`guard/train.py` checks that again).

In [ ]:
import hashlib, json, os, subprocess
OUT = "corpus/guardset/out"
assert EXPECTED_REPORT_SHA, "EXPECTED_REPORT_SHA is empty: the training set is not reviewed yet (Gate G2). Stop here."
if not os.path.exists(f"{OUT}/report.json"):
    tgz = f"{DRIVE_DIR}/guardset_out.tgz"
    if DATASET_REPO:
        from huggingface_hub import snapshot_download
        snapshot_download(DATASET_REPO, repo_type="dataset", revision=DATASET_REVISION or None, local_dir=OUT,
                          allow_patterns=["train.jsonl", "validation.jsonl", "report.json"])
    elif os.path.exists(tgz):
        subprocess.run(["tar", "xzf", tgz, "-C", REPO_DIR], check=True)
    else:
        raise SystemExit("no training set: set DATASET_REPO, or put guardset_out.tgz in " + DRIVE_DIR)
sha = hashlib.sha256(open(f"{OUT}/report.json", "rb").read()).hexdigest()
assert sha == EXPECTED_REPORT_SHA, f"report.json is {sha}, not the reviewed build {EXPECTED_REPORT_SHA}"
rep = json.load(open(f"{OUT}/report.json"))
print("training set OK:", json.dumps(rep["counts"], indent=1)[:1500])

## 6 · Runs

Per run: train (`guard/train.py`), score the frozen v2 set with it (`scripts/guard_eval.py --label
bhasha_guard@<run>`), save the log and scores to Drive, keep or delete the weights. A run whose
results are in Drive is skipped. Roughly 10-25 minutes per run on a T4.

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
LANGS = ["en", "hi", "hi_latn", "mixed"]

def results_of(name):
    return [f"guard/runs/{name}/train_log.json", f"guard/runs/{name}/val_scores.jsonl"] + \
           [f"analysis/out/guard_eval/v2/{l}__bhasha_guard@{name}.json" for l in LANGS]

def restore(name):
    src = f"{DRIVE_DIR}/results/{name}"
    if not os.path.isdir(src):
        return False
    for f in results_of(name):
        if os.path.exists(f"{src}/{os.path.basename(f)}"):
            os.makedirs(os.path.dirname(f), exist_ok=True)
            shutil.copy(f"{src}/{os.path.basename(f)}", f)
    return all(os.path.exists(f) for f in results_of(name))

def keep_weights(name):
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    src, dst = f"guard/runs/{name}/model", f"{DRIVE_DIR}/weights/{name}"
    m = AutoModelForSequenceClassification.from_pretrained(src).half()   # fp16: half the Drive space
    m.save_pretrained(dst + ".tmp", safe_serialization=True)
    AutoTokenizer.from_pretrained(src).save_pretrained(dst + ".tmp")
    shutil.rmtree(dst, ignore_errors=True)
    os.replace(dst + ".tmp", dst)

done, failed = [], []
for model in MODELS:
    for ablation in ABLATIONS:
        for seed in SEEDS:
            name = f"{model}__{ablation}__s{seed}"
            if restore(name):
                print("done already:", name); done.append(name); continue
            print(f"\n=== {name} ===", time.strftime("%H:%M:%S"))
            shutil.rmtree(f"guard/runs/{name}", ignore_errors=True)   # a half-finished run starts over
            rc = run([sys.executable, "-m", "guard.train", "--config", f"guard/configs/{model}.yaml",
                      "--seed", str(seed), "--ablation", ablation])
            if rc == 0:
                rc = run([sys.executable, "-m", "scripts.guard_eval", "--set", "v2", "--defenses", "bhasha_guard",
                          "--label", f"bhasha_guard@{name}", "--device", "0"],
                         env={"AGENTFW_GUARD_MODEL": f"guard/runs/{name}"})
            if rc != 0:
                print("FAILED:", name, "(re-run this cell to retry it)"); failed.append(name); continue
            log = json.load(open(f"guard/runs/{name}/train_log.json"))
            log["notebook_commit"] = COMMIT
            json.dump(log, open(f"guard/runs/{name}/train_log.json", "w"), indent=1)
            if seed in KEEP_WEIGHTS_SEEDS:
                keep_weights(name)
            os.makedirs(f"{DRIVE_DIR}/results/{name}", exist_ok=True)
            for f in results_of(name):
                shutil.copy(f, f"{DRIVE_DIR}/results/{name}/")
            shutil.rmtree(f"guard/runs/{name}/model", ignore_errors=True)
            done.append(name)
print("\ndone:", len(done), "| failed:", failed)

## 7 · Results archive

Logs and test-set scores of every finished run (no weights). Bring it back to the laptop.

In [ ]:
import glob, subprocess
files = sorted(glob.glob("guard/runs/*/train_log.json") + glob.glob("guard/runs/*/val_scores.jsonl")
               + glob.glob("analysis/out/guard_eval/v2/*__bhasha_guard@*.json"))
subprocess.run(["tar", "czf", "/content/guard_results.tgz", *files], check=True)
subprocess.run(["cp", "/content/guard_results.tgz", f"{DRIVE_DIR}/guard_results.tgz"], check=True)
print(len(files), "files ->", "guard_results.tgz (also in Drive)")
for f in sorted(glob.glob("guard/runs/*/train_log.json")):
    l = json.load(open(f)); print(f"{l['run']:<36} val PR-AUC {l['val_pr_auc']:.4f}  {l['seconds']:.0f}s")
try:
    from google.colab import files as colab_files
    colab_files.download("/content/guard_results.tgz")
except Exception:
    print("download /content/guard_results.tgz from the file browser or Drive")